# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Workflow Summary (per location)**
1. Extract annual maxima → fit stationary & non-stationary GEV per model
2. Compute return levels & CIs per year per model
3. Compute exceedance probability & CI for thresholds of interest
4. Aggregate multi-model ensemble: mean + total spread
5. Visualization
   1. Return levels vs year (shaded CI)
   2. Probability amplification vs year (shaded CI)
   3. Multi-model ensemble bar plots for future RL
   4. Maps (mean & spread)
6. Tables:
   1. RL per T per year ± CI
   2. Probability change of historical RL

# Import Libraries

In [ ]:
import sys
import random
import time
from datetime import datetime
from glob import glob
import warnings
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from pandas import DataFrame
from scipy.stats import genextreme
from numpy import array, mean, min, max, sqrt, log, arange
from scipy.stats import norm
import multiprocessing as mp
from joblib import Parallel, delayed
from tqdm import tqdm

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

# Settings

In [ ]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [ ]:
hindcast_start = 1960
hindcast_end = 2026

In [ ]:
# adding a little randomness to enhance trust the model works as robust as possible cross locations..
start_location = random.choice(arange(0, 9579))
end_location = start_location+10

print(f'analyse a subsample of location {start_location}–{end_location}')

In [ ]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [ ]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [ ]:
_LOCATION_LABELS = None

In [ ]:
dic_timing = {}
dic_notes_analysis = {}

In [ ]:
print_msg = True

export_report=True
display_results = False
save_regression_summary = True

# Import data

In [ ]:
dic_timing['import data'] = {}
dic_timing['import data']['start'] = datetime.now()

ls_files = [file for file in glob(path + '*.nc')]
ls_files

In [ ]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

dic_timing['import data']['end'] = datetime.now()

# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, using `joblib` - saving ~60% (from 3min30sec down to 1min22sec)

In [ ]:
dic_data_per_model, combined, notes_overview = dbf.prepare_combined_data(ls_files, dic_data_per_model)
dic_notes_analysis['data overview'] = notes_overview

## Crop data to hindcast period and rearrange to per location

Full data set requires 9589 tasks to be completed ~1min

In [ ]:
dic_data_per_location = dbf.extract_location_data(combined, hindcast_start, hindcast_end)

In [ ]:
if start_location is not None or end_location is not None: 
    dic_data_per_location = ut.select_allowed_locations(
        dic_data_per_location=dic_data_per_location, 
        start_loc=start_location, end_loc=end_location
        )
    print(f'Processing locations {start_location} to {end_location} ({len(dic_data_per_location)} total)')

else:
    print(f'Processing all {len(dic_data_per_location)} locations')

print('Getting closest point available as location label for orientation. \nNote this is not the exact location...')
location_labels = dbf.precompute_location_labels(dic_data_per_location)
location_labels

## Validation Check

In [ ]:
dic_timing['data validity check'] = {}
dic_timing['data validity check']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
list_model_labels = list(dic_data_per_model.keys())

model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

# Workflow GEV - Generalized Extreme Value

## OPTION1
Using all data available per location - from all years (within hind-cast period) and models.

**Run Analysis**<br>

Note, the output is stored as 
- visuals → png
- tabular data (DataFrames) → Parquet
- other objects (dicts, strings, floats) → Pickle

File structure
```results/
├─ location_1/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
├─ location_2/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
...
```

In [ ]:
def process_location(location_item, return_periods)->dict:
    loc_id, df_prepared = location_item
    messages = []

    lon_loc = df_prepared.lon.unique()[0]
    lat_loc = df_prepared.lat.unique()[0]

    location_info = _LOCATION_LABELS.get((round(lon_loc,6), round(lat_loc,6)), "unknown location")

    result, ls_warnings = gev.analyze_per_location(
        df_prepared, loc_id, lat_loc, lon_loc, location_info, return_periods
    )
    if ls_warnings:
        messages.append({loc_id: ls_warnings})
    
    if result is None:
        messages.append(f"No valid GEV fit for location {loc_id}")
        return loc_id, None, messages

    return {
        "loc_id": loc_id,
        "data": df_prepared,
        "result": result,
        "location_info": {"lat": lat_loc, "lon": lon_loc,"label": location_info},
        "messages": messages,
    }

In [ ]:
print(f'Run GEV analysis with pooled data for locations {list(dic_data_per_location.keys())}...')

# ----------------------------------------------------------------------------------------------------
def set_location_labels(labels):
    global _LOCATION_LABELS
    _LOCATION_LABELS = labels
    
# ----------------------------------------------------------------------------------------------------
set_location_labels(location_labels)

out = Parallel(n_jobs=-1, backend='loky', verbose=10)(
    delayed(process_location)(item, return_periods) 
    for item in list(dic_data_per_location.items())
    )
    
all_data = []
results = {}
location_info = {}
ls_notes = []
for res in out:
    if res is None:
        continue

    loc_id = res["loc_id"]
    all_data.append(res["data"].assign(location_id=loc_id))
    results[loc_id] = res['result']
    location_info[loc_id] = res["location_info"]

    if res.get("messages"):
        ls_notes.extend(res["messages"])

# ----------------------------------------------------------------------------------------------------
output = {"data": all_data, "results": results, "location_info": location_info,}
results = output['results']  
dic_notes_analysis['GEV pooled analysis'] = ls_notes

In [ ]:
dic_figures = {}
for loc_id, result in results.items():
    fig = dbplt.plot_pooled_analysis(
        result=result, site_id=loc_id, 
        periods_evolution=plot_period_evolution, 
        leg_comparison_x=0.35, leg_comparison_y=0.65, 
        linestyle_trends=['dashdot', 'dashed', 'solid'], 
        fontsize=12, figsize=(15, 7.5)
        )
    dic_figures[loc_id] = fig

In [ ]:
dic_figures[random.choice(list(results.keys()))]

## OPTION2
Re-run stationary GEV per year (for location parameter; scale and shape remain as globally defined)

In [ ]:
# del results
try:
    results.keys()
    print('✓ continue with available dictionary')
    
except NameError:
    print('import data from files...')
    results = ut.load_pooled_results('../output/gev_analysis/pooled/2026-02-06/')

In [ ]:
site_ex = random.choice(list(results.keys()))
print(f'annual stationary GEV on site: {site_ex}')

In [ ]:
results[site_ex].keys()

In [ ]:
print(results[site_ex]['fit results'].keys())

print('\nGEVstationary')
print(results[site_ex]['fit results']['gev_stationary'].keys())

In [ ]:
column_label_year = None
for c in results[site_ex]['data'].columns:
    if 'year' in c:
        column_label_year = c


if not column_label_year:
    print('WARNING - not year column identified. Please check data!')
    
else:
    results_extended, ls_notes_analysis = gev.execute_and_store_stat_gev_per_year(
        results=results, return_periods=return_periods, 
        column_label_year=column_label_year, store_results=False, 
        )

    # ------------------------------------------------------------------------------------------
    for key, outer_list in ls_notes_analysis.items():
        ls_notes_analysis[key] = [inner for inner in outer_list if inner]
    dic_notes_analysis['annual_statGEV'] = ls_notes_analysis


In [ ]:
print(results_extended[site_ex].keys())

print(results_extended[site_ex]['fit results'].keys())
print('\nGEVstationary')
print(results_extended[site_ex]['fit results']['gev_stationary'].keys())

# Subsequent Analysis

## Regression curve for location parameter
including uncertainty given by n_obs

**NOTE**<br>
> centering the year parameter due to the following warning:<br>
*"The condition number is large, 2.37e+05. This might indicate that there are strong multicollinearity or other numerical problems."*

In [ ]:
df_annual_fit = results_extended[site_ex]['fit results']['gev_stationary']['analysis_per_year']
df_annual_fit

In [ ]:
plt.plot(df_annual_fit.location)

In [ ]:
dic_timing = {}

In [ ]:
path_results = '../output/gev_analysis/pooled/2026-02-06/'

In [ ]:
results = ut.load_fit_results(path_results)
results = ut.select_allowed_locations(dic_data_per_location=results, start_loc=0, end_loc=10)

results.keys()

In [ ]:
results1 = []
for site_id, location_results in results.items():
        df = gev.weighted_least_square_regression_for_site_mp(site_id,location_results)
        results1.append(df)
        
for site_id, analysis_dict in results1:
        results.setdefault(site_id, {}).update(analysis_dict)

In [ ]:
%matplotlib inline 

result_key = list(results.keys())[0]
column_label = None
for col_label in results[result_key]['data'].columns:
    if col_label in ('sim_year', 'year'):
        column_label = col_label
    
if column_label is None:
    raise ValueError('No year found in data table')

site_id = 7 
dic_location = results[site_id]
wls_delta = dic_location['wls_delta']

print(column_label)

# -------------------------
df=dic_location['df']
weights=dic_location['weights']
year_grid=dic_location['year_grid']
year_mean=dic_location['year_mean']
y_pred=dic_location['y_pred']
lat=dic_location['data']['lat'].unique()[0]
lon=dic_location['data']['lon'].unique()[0]

# -------------------------

# annual stationary GEV analysis
intercept, slope = wls_delta.params

mask = df['location'].notna() & df['var_mu'].notna()
df_plot = df[mask].copy()
df_plot['location'] = df_plot.location *1000
weights_plot = array(weights)[mask]

cov = wls_delta.cov_params().values  
t_centered = year_grid - year_mean
pred_var_t = cov[0,0] + t_centered**2 * cov[1,1] + 2 * t_centered * cov[0,1]
pred_std_t = sqrt(pred_var_t)
y_upper = y_pred + 1.96 * pred_std_t
y_lower = y_pred - 1.96 * pred_std_t


In [ ]:
df_plot

In [ ]:
import matplotlib.pyplot as plt

figsize = (13, 3.5)
fontsize: float = 11
figsize: tuple[float, float] = (13, 3.5)
axes_color: str = '#333333'
markers_color: str = "#99E3DDFF",
colors_reg: list = ['#333333FF', '#7F6C7BFF']  

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

ax.scatter(
    df_plot['year'], df_plot['location'], s=weights_plot/1000, 
    alpha=0.6, c=markers_color, label=r'Annual estimates $\mu$'
) 
for _, row in df_plot.iterrows():
    ax.text(row.year, row.location, f"{row.n_obs}", fontsize=8, alpha=0.6)

ax.plot(
    year_grid, y_pred, color='black', 
    label=f'Annual stationary μ(t)\nslope={slope:.5f}, intercept={intercept:.4f} (centered {int(year_mean)})'
    )
ax.fill_between(year_grid, y_lower, y_upper, color=colors_reg[0], alpha=0.15, label='95% CI (annual stationary)')
    
    
leg = ax.legend(loc=0, edgecolor=axes_color, borderpad=.65, fontsize=fontsize*0.75)
leg.get_frame().set_linewidth(.5)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.axhline(y=ax.get_ylim()[0], color=axes_color, linewidth=1.2, zorder=10)
ax.axvline(x=ax.get_xlim()[0], color=axes_color, linewidth=1.2, zorder=10)

ax.tick_params(axis='x', colors=axes_color)
ax.tick_params(axis='y', colors=axes_color)

ax.grid(True, alpha=0.3, color='lightgrey')
ax.set_title(
    f'GEV μ Trend with Fixed Scale & Shape for lat|lon {lat:.3f}|{lon:.3f} (siteID {site_id})', 
    fontsize=fontsize*1.25
    )
ax.set_xlabel('Year', fontsize=fontsize)
ax.set_ylabel(r'GEV fitted $\mu$, in mm/yr', fontsize=fontsize)

plt.tight_layout()
fig.canvas.draw()

In [ ]:

fig = dbplt.plot_gev_mu_trend(
        df=dic_location['df'],
        weights=dic_location['weights'],
        year_grid=dic_location['year_grid'],
        year_mean=dic_location['year_mean'],
        y_pred=dic_location['y_pred'],
        wls_delta=wls_delta,
        nonstat_years=dic_location['data'][column_label].values.astype(int), 
        nonstat=dic_location['fit results']['gev_nonstationary'],
        display_results=True,
        colors_reg=['#333333FF', '#C88D35FF'],
        markers_color='#99E3DDFF',
        lat=dic_location['data']['lat'].unique()[0],
        lon=dic_location['data']['lon'].unique()[0],
        site_id=site_id,
)


In [ ]:
dic_timing['regression start'] = datetime.now()
save_regression_summary = False
# ------------------------------------------------------------------------------------------
result_key = list(results.keys())[0]
column_label = None
for col_label in results[result_key]['data'].columns:
    if col_label in ('sim_year', 'year'):
        column_label = col_label
    
if column_label is None:
    raise ValueError('No year found in data table')

for site_id, dic_location in results.items():
    wls_delta = dic_location['wls_delta']

    fig = dbplt.plot_gev_mu_trend(
            df=dic_location['df'],
            weights=dic_location['weights'],
            year_grid=dic_location['year_grid'],
            year_mean=dic_location['year_mean'],
            y_pred=dic_location['y_pred'],
            wls_delta=wls_delta,
            nonstat_years=dic_location['data'][column_label].values.astype(int), 
            nonstat=dic_location['fit results']['gev_nonstationary'],
            display_results=True,
            colors_reg=['#333333FF', '#C88D35FF'],
            markers_color='#99E3DDFF',
            lat=dic_location['data']['lat'].unique()[0],
            lon=dic_location['data']['lon'].unique()[0],
            site_id=site_id,
    )

    if save_regression_summary and ('file location' in dic_location.keys() or 'file_path_report' in dic_location.keys()):
        if 'file location' in dic_location.keys():
            save_path = dic_location['file location']
        else:
            save_path = dic_location['file_path_report']
        with open(save_path + '/WLSdelta_summary.html', 'w') as f:
            f.write( wls_delta.summary().as_html())
        
        lat = str(dic_location['location info']['lat'].round(3))
        lon = str(dic_location['location info']['lon'].round(3))
        file_name = f"/GEVTrendAnalysis_location_{str(site_id)}_{lat}|{lon}.png"
        fig.savefig(save_path+file_name, dpi=300, bbox_inches='tight')

    else:
        print("\t skipping saving GEV μ trend analysis ...")

# ------------------------------------------------------------------------------------------
dic_timing['regression end'] = datetime.now()
time_diff = dic_timing['regression end'] - dic_timing['regression start']
print(
    f"\nExecution time for computing regression analysis for annual stationary GEV and non-stationary GEV: "
    f"{time_diff}sec"
    )

In [ ]:
ut.store_analysis_notes(dic_notes_analysis, path_export)

## Future Return Levels based on Regression for LocationParameter

#### Utils

In [ ]:
from typing import Optional, Union
import func_plotting as dbplti
import func_utils as ut
import statsmodels.api as sm
from numpy import (any, array, exp, finfo, float64, full_like, generic, inf,
                   isfinite, isnan, linalg, linspace, log, nan, ndarray,
                   ones_like, sqrt, sum, vstack, zeros)
from pandas import DataFrame, to_numeric, read_html
from scipy import optimize, stats
from scipy.optimize import approx_fprime, minimize
from scipy.stats import norm
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from pathlib import Path


import logging
import os
import pickle
import re
import sys
from datetime import datetime
from glob import glob
from logging import FileHandler, Logger
from pathlib import Path
from typing import Optional, Union

import arabic_reshaper
import func_plotting as dbplt
from bidi.algorithm import get_display
from bs4 import BeautifulSoup
from joblib import Parallel, delayed
from numpy import isnan, unique, diag
from pandas import DataFrame, concat, read_parquet


In [ ]:
def project_gev_params_to_year(gev_nonstat, year, year_ref):
    """
    Project non-stationary GEV parameters to a given year.
    Assumes linear trend in location only.
    """
    
    mu0, mu1 = gev_nonstat['mu0'], gev_nonstat['mu1']
    sigma, xi = gev_nonstat['sigma'], gev_nonstat['xi']

    mu_t = mu0 + mu1 * (year - year_ref)

    return {
        'mu': mu_t,
        'sigma': sigma,
        'xi': xi
    }


def return_levels_at_year(gev_nonstat, return_periods, year, year_ref):
    """
    Return levels for a non-stationary GEV evaluated at a specific year.
    """
    params = project_gev_params_to_year(gev_nonstat, year, year_ref)

    p = 1 - 1 / array(return_periods)

    rl = genextreme.ppf(
        p,
        c=-params['xi'],
        loc=params['mu'],
        scale=params['sigma']
    )

    return dict(zip(return_periods, rl))


def return_level_ci(gev_nonstat, T, year, cov_matrix, year_ref, alpha=0.05):

    mu0, mu1 = gev_nonstat['mu0'], gev_nonstat['mu1']
    sigma, xi = gev_nonstat['sigma'], gev_nonstat['xi']

    var_mu0 = cov_matrix.loc['mu0','mu0']
    var_mu1 = cov_matrix.loc['mu1','mu1']
    cov_mu0_mu1 = cov_matrix.loc['mu0','mu1']

    t = year - year_ref
    var_mu_t = var_mu0 + t**2 * var_mu1 + 2*t*cov_mu0_mu1
    sd_mu_t = sqrt(var_mu_t)

    zT = mu0 + mu1*t + (sigma/xi) * ((-log(1-1/T))**(-xi)-1)

    zT_lower = zT - norm.ppf(1-alpha/2)*sd_mu_t
    zT_upper = zT + norm.ppf(1-alpha/2)*sd_mu_t

    return zT, (zT_lower, zT_upper)


def ensemble_return_levels(models_results, T, year, year_ref):
    zT_list = []
    zT_lower_list = []
    zT_upper_list = []

    for res in models_results:
        gev = res['gev_nonstationary']
        cov = res['cov_matrix']
        zT, (zl, zu) = return_level_ci(gev, T, year, cov, year_ref)
        zT_list.append(zT)
        zT_lower_list.append(zl)
        zT_upper_list.append(zu)

    ensemble_mean = mean(zT_list)
    ensemble_lower = min(zT_lower_list)
    ensemble_upper = max(zT_upper_list)

    return ensemble_mean, (ensemble_lower, ensemble_upper)


In [ ]:
def load_fit_results(base_dir: Union[str, Path], wls_file: Optional[str]=None) -> dict[int, dict[str, any]]:
    """
    Load pooled results stored in artifact-centric format:
      - DataFrames: *.parquet (with location_id column)
      - Python objects: *.pkl (dict keyed by location_id)

    Returns:
        results: Dict[location_id -> dict of artifact_name -> artifact]
    """
    base_dir = Path(base_dir)
    results: dict[int, dict[str, any]] = {}

    for pickle_file in base_dir.glob("*.pkl"):
        key = pickle_file.stem
        if key == 'fit results':
            with open(pickle_file, "rb") as f:
                obj = pickle.load(f)
                for loc_id, value in obj.items():
                    results.setdefault(int(loc_id), {})[key] = value
    
    for parquet_file in base_dir.glob("*.parquet"):
        key = parquet_file.stem
        df = read_parquet(parquet_file)
        if "location_id" in df.columns:
            for loc_id, df_loc in df.groupby("location_id"):
                results.setdefault(int(loc_id), {})[key] = df_loc.drop(columns="location_id")
        else:
            results.setdefault(0, {})[key] = df

    if wls_file:
        html_file = base_dir / wls_file
        print(html_file)

        with open(html_file, 'r') as f:
            html_text = f.read()

        chunks = html_text.split('<h2>location ')
        for chunk in chunks[1:]:  
            loc_id_str, content = chunk.split('</h2>', 1)
            loc_id = int(loc_id_str.strip())

            if '<h2>' in content:
                content, _ = content.split('<h2>', 1)
            
            results.setdefault(loc_id, {})['WLSdelta_summary'] = content
        
    else:
        print(f'Could not file WLS file under {html_file}')
    return results

#### (Re-) import data and results 
Selecting a subset of 10 locations

In [ ]:
year_future_risk = 2050

In [ ]:
path_results = '../output/gev_analysis/pooled/2026-02-06/'

In [ ]:
results = load_fit_results(path_results, wls_file='WLSdelta_summary.html')

In [ ]:
results = ut.select_allowed_locations(dic_data_per_location=results, start_loc=0, end_loc=10)
results.keys()

In [ ]:
results[0].keys()

##### Compute Future Risk

In [ ]:
rl_2050 = return_levels_at_year(
    results[0]['fit results']['gev_nonstationary'],
    return_periods=[10, 20, 50, 100],
    year=year_future_risk,
    year_ref=results[0]['data'].sim_year.mean()
)

rl_2050

In [ ]:
def compute_cov_matrix(gev_params: dict, data: ndarray, years: ndarray = None, annual_stat:Optional[bool]=False) -> ndarray:
    """
    Compute the covariance matrix of fitted GEV parameters using numerical Hessian.

    Parameters
    ----------
    gev_params : dict
        Fitted GEV parameters. Should contain:
        - 'shape', 'location', 'scale' for stationary
        - or 'mu0', 'mu1', 'sigma', 'xi' for non-stationary location trend
    data : np.ndarray
        Observed annual maxima used for fitting
    years : np.ndarray, optional
        Years array (required if non-stationary model with trend)

    Returns
    -------
    cov_matrix : np.ndarray
        Covariance matrix of parameters
    """
    
    # Define parameter vector theta and negative log-likelihood
    if 'trend_in' in gev_params and gev_params['trend_in'] == 'location':
        # Non-stationary μ(t) = μ0 + μ1 * t
        t = (years - gev_params['years_mean']) / gev_params['years_std']

        def neg_loglik(theta):
            mu0, mu1, sigma, xi = theta
            mu_t = mu0 + mu1 * t
            c = -xi  # SciPy convention
            return -sum(stats.genextreme.logpdf(data, c, loc=mu_t, scale=sigma))
        theta_hat = array([gev_params['mu0'], gev_params['mu1'], gev_params['sigma'], gev_params['xi']])
    elif annual_stat:
        print('compute annual stationary case')
        t = (years - array(years).mean()) / array(years).std()

        def neg_loglik(theta):
            mu0, mu1, sigma, xi = theta
            mu_t = mu0 + mu1 * t
            c = -xi  # SciPy convention
            return -sum(stats.genextreme.logpdf(data, c, loc=mu_t, scale=sigma))
        theta_hat = array([gev_params['mu0'], gev_params['mu1'], gev_params['sigma'], gev_params['xi']])
        
    else:
        # Stationary GEV: μ, σ, ξ
        def neg_loglik(theta):
            xi, mu, sigma = theta
            c = -xi
            return -sum(stats.genextreme.logpdf(data, c, loc=mu, scale=sigma))

        theta_hat = array([gev_params['shape'], gev_params['location'], gev_params['scale']])

    epsilon = sqrt(finfo(float).eps)
    def grad(theta):
        return optimize.approx_fprime(theta, neg_loglik, epsilon)

    H = optimize.approx_fprime(theta_hat, grad, epsilon)

    # Covariance = inverse of Hessian
    try:
        cov_matrix = linalg.inv(H)
    except linalg.LinAlgError:
        logger.info("Warning: Hessian not invertible; returning None")
        return None
    
    return cov_matrix



In [ ]:
results_location = results[0]

results_location.keys()

In [ ]:
results_location['fit results']['gev_stationary'].keys()

In [ ]:
results_location['fit results']['gev_nonstationary'].keys()

In [ ]:
tables = read_html(results_location['WLSdelta_summary'])
coef_table = tables[1].set_index(0)
coef_table = coef_table.T.set_index(coef_table.T.columns[0]).T
coef_table

In [ ]:
std_err = coef_table['std err'].to_numpy().astype(float)
cov_matrix = diag(std_err**2)
cov_matrix

In [ ]:
cov = wls_delta.cov_params().values  
t_centered = year_grid - year_mean
pred_var_t = cov[0,0] + t_centered**2 * cov[1,1] + 2 * t_centered * cov[0,1]
pred_std_t = sqrt(pred_var_t)
y_upper = y_pred + 1.96 * pred_std_t
y_lower = y_pred - 1.96 * pred_std_t

In [ ]:
# annual stat
cov_annual_stationary = compute_cov_matrix(
    gev_params=results_location['fit results']['gev_stationary']['analysis_per_year'], 
    data=results_location['data']['storm_surge'],
    years=results_location['fit results']['gev_stationary']['analysis_per_year'].index,
    annual_stat=True
    )

# non-stat
#cov_nonstat = compute_cov_matrix(
#    gev_params=results_location['fit results']['gev_nonstationary'], 
#    data=results_location['data']['storm_surge'], 
#    years=results_location['data']['sim_year']
#    )

# stationary
#cov_stationary = compute_cov_matrix(
#    gev_params=results_location['fit results']['gev_stationary'], 
#    data=results_location['data']['storm_surge']
#    )

#gev.calculate_return_level(gev_params, return_periods, year=year_future_risk)

In [ ]:
results_location['fit results']['gev_nonstationary']

In [ ]:
results_location['fit results']['gev_stationary']['analysis_per_year']

## Time-Varying Exceedance Probability

<i> What is the probability in 2050 of exceeding the 1960 50-year level? <i>

In [ ]:
def exceedance_probability(gev_nonstat, threshold, year, year_ref):
    params = project_gev_params_to_year(gev_nonstat, year, year_ref)

    cdf = genextreme.cdf(
        threshold,
        c=-params['xi'],
        loc=params['mu'],
        scale=params['sigma']
    )

    return 1 - cdf


In [ ]:
z50_1960 = rl_nonstat_start[50] 

p_2050 = exceedance_probability(
    gev_nonstat_loc,
    threshold=z50_1960,
    year=2050,
    year_ref=years.mean()
)


In [ ]:
def analyze_per_location_future(
    data_hindcast,
    site_id,
    lat,
    lon,
    location_info,
    return_periods,
    future_years=(2030, 2050)
):
    results, notes = analyze_per_location(
        data_hindcast,
        site_id,
        lat,
        lon,
        location_info,
        return_periods
    )

    gev_nonstat = results['fit results']['gev_nonstationary']
    years = results['data']['year'].values
    year_ref = years.mean()

    future_rl = {}
    future_probs = {}

    if gev_nonstat:
        # reference level: 50y at start
        z50_ref = results['return_levels']['nonstationary_start']['values'][50]

        for y in future_years:
            future_rl[y] = return_levels_at_year(
                gev_nonstat, return_periods, y, year_ref
            )

            future_probs[y] = exceedance_probability(
                gev_nonstat, z50_ref, y, year_ref
            )

    results['future'] = {
        'return_levels': future_rl,
        'exceedance_probabilities': future_probs,
        'reference_level': z50_ref
    }

    return results, notes


## Multi-model Comparison

Alternative: sample μ0, β from all models → compute return level distribution → percentile CI

Then you can plot spatial maps of ensemble mean & spread across 11,000 locations

# Visualizations

- Time-varying return levels (line + shaded CI)
- Time-varying exceedance probability of historical extremes (line + CI)
- Ensemble mean return levels at future years (bar + error bars)
- Spatial maps (Europe-wide):
    - Mean RL at 2050
    - Change in probability vs baseline (1960)
    - Spread across models

### Return Level Evolution

In [ ]:
def plot_return_level_evolution(gev_nonstat, return_periods, years, year_ref):
    import matplotlib.pyplot as plt

    for T in return_periods:
        rl_t = [
            return_levels_at_year(gev_nonstat, [T], y, year_ref)[T]
            for y in years
        ]
        plt.plot(years, rl_t, label=f'{T}-year')

    plt.xlabel('Year')
    plt.ylabel('Storm surge (m)')
    plt.legend()
    plt.title('Time-varying return levels')
    plt.grid(True)


### Probability Amplification

In [ ]:
def plot_probability_change(gev_nonstat, threshold, years, year_ref):
    import matplotlib.pyplot as plt

    probs = [
        exceedance_probability(gev_nonstat, threshold, y, year_ref)
        for y in years
    ]

    plt.plot(years, probs)
    plt.ylabel('Annual exceedance probability')
    plt.xlabel('Year')
    plt.title('Increasing exceedance probability of historical extreme')
    plt.grid(True)



#### Potential Additional Visualizations
- Maps of 100-year return levels along European coastline
- Difference maps: non-stationary minus stationary → climate change impact
- Probability exceedance curves for selected cities
- Histograms / density of return levels → compare regions
- Time series of non-stationary μ or return levels → show increasing trends